In [0]:
import builtins
builtins_round = builtins.round

In [0]:
import logging
import builtins
from pyspark.sql.functions import col, when, lit, round as spark_round, rank, desc, count, sum, avg, to_date, year, month, hour, dayofweek, current_timestamp, current_date, datediff, monotonically_increasing_id, nullif
from pyspark.sql.window import Window

logging.basicConfig(level=logging.INFO)
logger = logging.getLogger("gold_layer")
builtins_round = builtins.round

try:
    tweets    = spark.table("social_catalog.silver.tweets_table")
    sentiment = spark.table("social_catalog.silver.sentiment_table")
    trends    = spark.table("social_catalog.silver.trends_table")
    users     = spark.table("social_catalog.silver.user_metadata_table")
    valid     = spark.table("social_catalog.silver.valid_table")

    spark.sql("CREATE SCHEMA IF NOT EXISTS social_catalog.gold")

    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.dim_topic")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.dim_country")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.dim_user")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.fact_tweet")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.fact_trend")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.agg_sentiment_by_topic")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.agg_user_influence")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.tweet_metrics")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.tweets_per_day")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.tweets_per_hour")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.sentiment_metrics")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.sentiment_over_time")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.trend_metrics")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.trend_over_time")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.user_metrics")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.top_users_by_followers")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.user_creation_trend")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.users_by_country")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.valid_tweet_metrics")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.valid_tweets_per_day")
    spark.sql("DROP TABLE IF EXISTS social_catalog.gold.valid_tweets_per_topic")

    logger.info("All gold tables dropped")

    # dim_topic
    dim_topic = (
        sentiment.select(col("topic_category").alias("topic_name"))
        .union(users.select(col("topic_category").alias("topic_name")))
        .union(valid.select(col("topic_category").alias("topic_name")))
        .filter(col("topic_name").isNotNull())
        .dropDuplicates()
        .withColumn("topic_id", monotonically_increasing_id())
        .withColumn("created_at", current_timestamp())
        .select("topic_id", "topic_name", "created_at")
    )
    dim_topic.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.dim_topic")
    logger.info("dim_topic : " + str(spark.table("social_catalog.gold.dim_topic").count()) + " rows")

    # dim_country
    dim_country = (
        trends.select(col("country").alias("country_name"))
        .union(users.select(col("country").alias("country_name")))
        .filter(col("country_name").isNotNull())
        .dropDuplicates()
        .withColumn("country_id", monotonically_increasing_id())
        .select("country_id", "country_name")
    )
    dim_country.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.dim_country")
    logger.info("dim_country : " + str(spark.table("social_catalog.gold.dim_country").count()) + " rows")

    dim_topic   = spark.table("social_catalog.gold.dim_topic")
    dim_country = spark.table("social_catalog.gold.dim_country")

    # dim_user - keep all rows
    dim_user = (
        users
        .join(dim_country, users["country"] == dim_country["country_name"], "left")
        .join(dim_topic,   users["topic_category"] == dim_topic["topic_name"], "left")
        .withColumn("follower_segment",
            when(col("followers_count") >= 100000, "Mega Influencer")
            .when(col("followers_count") >= 10000,  "Macro Influencer")
            .when(col("followers_count") >= 1000,   "Micro Influencer")
            .otherwise("Regular User")
        )
        .withColumn("account_age_days", datediff(current_date(), col("account_created_date").cast("date")))
        .withColumn("is_verified", when(col("varified") == "true", True).otherwise(False))
        .withColumn("likes_count",     when(col("likes_count").isNull(),     lit(0)).otherwise(col("likes_count")))
        .withColumn("following_count", when(col("following_count").isNull(), lit(0)).otherwise(col("following_count")))
        .withColumn("followers_count", when(col("followers_count").isNull(), lit(0)).otherwise(col("followers_count")))
        .withColumn("shares_count",    when(col("shares_count").isNull(),    lit(0)).otherwise(col("shares_count")))
        .withColumn("posts_count",     when(col("posts_count").isNull(),     lit(0)).otherwise(col("posts_count")))
        .select(
            "user_id", "country_id", "topic_id",
            "account_created_date", "account_age_days",
            "followers_count", "following_count",
            "likes_count", "shares_count", "posts_count",
            "is_verified", "follower_segment"
        )
    )
    dim_user.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.dim_user")
    logger.info("dim_user : " + str(spark.table("social_catalog.gold.dim_user").count()) + " rows")

    # fact_tweet - merge tweets + valid
    tweets_ready = (
        tweets
        .withColumnRenamed("timestamp", "tweet_timestamp")
        .withColumn("is_valid",         lit(False))
        .withColumn("retweets",         col("retweets").cast("bigint"))
        .withColumn("replies",          col("replies").cast("bigint"))
        .withColumn("likes",            col("likes").cast("bigint"))
        .withColumn("impressions",      col("impressions").cast("bigint"))
        .withColumn("engagement_count", col("engagement").cast("bigint"))
        .select(col("tweet_id"), col("user_id").cast("bigint"), col("tweet_text"),
                col("tweet_timestamp"), col("likes"), col("retweets"), col("replies"),
                col("impressions"), col("engagement_count"), col("is_valid"))
    )
    valid_ready = (
        valid
        .withColumn("is_valid",         lit(True))
        .withColumn("user_id",          lit(None).cast("bigint"))
        .withColumn("likes",            col("likes").cast("bigint"))
        .withColumn("retweets",         col("retweets").cast("bigint"))
        .withColumn("replies",          col("replies").cast("bigint"))
        .withColumn("impressions",      col("impressions").cast("bigint"))
        .withColumn("engagement_count", col("engagement_count").cast("bigint"))
        .select("tweet_id", "user_id", "tweet_text", "tweet_timestamp",
                "likes", "retweets", "replies", "impressions", "engagement_count", "is_valid")
    )

    merged_tweets = tweets_ready.unionByName(valid_ready)

    fact_tweet = (
        merged_tweets
        .join(sentiment.select("tweet_id", "topic_category", "sentiment_score",
                               "positive_score", "negative_score", "neutral_score"),
              on="tweet_id", how="left")
        .join(dim_topic, col("topic_category") == dim_topic["topic_name"], "left")
        .withColumn("engagement_rate_pct",
            spark_round(col("engagement_count") / nullif(col("impressions"), lit(0)) * 100, 2))
        .withColumn("sentiment_label",
            when(col("sentiment_score") >  0.05, "Positive")
            .when(col("sentiment_score") < -0.05, "Negative")
            .otherwise("Neutral"))
        .withColumn("tweet_date",  to_date(col("tweet_timestamp")))
        .withColumn("tweet_year",  year(col("tweet_timestamp")))
        .withColumn("tweet_month", month(col("tweet_timestamp")))
        .withColumn("tweet_hour",  hour(col("tweet_timestamp")))
        .withColumn("day_of_week", dayofweek(col("tweet_timestamp")))
        .select("tweet_id", "user_id", "topic_id", "tweet_text", "tweet_timestamp",
                "tweet_date", "tweet_year", "tweet_month", "tweet_hour", "day_of_week",
                "likes", "retweets", "replies", "impressions", "engagement_count",
                "engagement_rate_pct", "is_valid", "sentiment_score", "positive_score",
                "negative_score", "neutral_score", "sentiment_label")
    )
    fact_tweet.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.fact_tweet")
    logger.info("fact_tweet : " + str(spark.table("social_catalog.gold.fact_tweet").count()) + " rows")

    # fact_trend - keep all rows
    fact_trend = (
        trends
        .join(dim_country, trends["country"] == dim_country["country_name"], "left")
        .withColumn("trend_date", to_date(col("trend_timestamp")))
        .withColumn("trend_strength",
            when(col("trend_score") >= 0.75, "Viral")
            .when(col("trend_score") >= 0.50, "Trending")
            .when(col("trend_score") >= 0.25, "Growing")
            .otherwise("Low"))
        .withColumn("sentiment_category",
            when(col("sentiment_index") >  0.05, "Positive Trend")
            .when(col("sentiment_index") < -0.05, "Negative Trend")
            .otherwise("Neutral Trend"))
        .withColumn("country_rank",
            rank().over(Window.partitionBy("country_id", "trend_date").orderBy(desc("trend_score"))))
        .withColumn("is_top3", when(col("country_rank") <= 3, True).otherwise(False))
        .select("trend_timestamp", "trend_date", "country_id",
                "tweet_volume", "mention_count", "retweet_count",
                "trend_score", "sentiment_index", "impressions", "engagement_count",
                "trend_strength", "sentiment_category", "country_rank", "is_top3")
    )
    fact_trend.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.fact_trend")
    logger.info("fact_trend : " + str(spark.table("social_catalog.gold.fact_trend").count()) + " rows")

    # agg_sentiment_by_topic - group by tweet_id level for more rows
    agg_sentiment_by_topic = (
        sentiment
        .join(dim_topic, sentiment["topic_category"] == dim_topic["topic_name"], "left")
        .withColumn("sentiment_date", to_date(col("tweet_timestamp")))
        .withColumn("sentiment_hour", hour(col("tweet_timestamp")))
        .groupBy("topic_id", "topic_name", "sentiment_date", "sentiment_hour")
        .agg(
            count("tweet_id").alias("total_tweets"),
            spark_round(avg("sentiment_score"), 4).alias("avg_sentiment_score"),
            spark_round(avg("positive_score"),  4).alias("avg_positive_score"),
            spark_round(avg("negative_score"),  4).alias("avg_negative_score"),
            spark_round(avg("neutral_score"),   4).alias("avg_neutral_score"),
            sum("impressions").alias("total_impressions"),
            sum("likes").alias("total_likes"),
            sum("engagement_count").alias("total_engagement")
        )
        .withColumn("dominant_sentiment",
            when(col("avg_positive_score") > col("avg_negative_score"), "Positive")
            .when(col("avg_negative_score") > col("avg_positive_score"), "Negative")
            .otherwise("Neutral"))
    )
    agg_sentiment_by_topic.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.agg_sentiment_by_topic")
    logger.info("agg_sentiment_by_topic : " + str(spark.table("social_catalog.gold.agg_sentiment_by_topic").count()) + " rows")

    # agg_user_influence - keep all users
    fact_tweet_loaded = spark.table("social_catalog.gold.fact_tweet")
    dim_user_loaded   = spark.table("social_catalog.gold.dim_user")

    user_tweet_stats = (
        fact_tweet_loaded.groupBy("user_id")
        .agg(
            count("tweet_id").alias("total_tweets"),
            sum("likes").alias("total_likes"),
            sum("retweets").alias("total_retweets"),
            sum("replies").alias("total_replies"),
            sum("impressions").alias("total_impressions"),
            sum("engagement_count").alias("total_engagement"),
            spark_round(avg("engagement_count"), 2).alias("avg_engagement")
        )
    )

    agg_user_influence = (
        dim_user_loaded
        .join(user_tweet_stats, on="user_id", how="left")
        .withColumn("influence_score",
            spark_round(
                (col("total_likes")    * 1.0) +
                (col("total_retweets") * 2.0) +
                (col("total_replies")  * 1.5), 2))
        .withColumn("global_rank",  rank().over(Window.orderBy(desc("influence_score"))))
        .withColumn("topic_rank",   rank().over(Window.partitionBy("topic_id").orderBy(desc("influence_score"))))
        .withColumn("country_rank", rank().over(Window.partitionBy("country_id").orderBy(desc("influence_score"))))
    )
    agg_user_influence.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.agg_user_influence")
    logger.info("agg_user_influence : " + str(spark.table("social_catalog.gold.agg_user_influence").count()) + " rows")

    # tweet_metrics
    total_tweets        = tweets.count()
    avg_likes_val       = tweets.agg(spark_round(avg("likes"), 2).alias("v")).collect()[0]["v"]
    avg_retweets_val    = tweets.agg(spark_round(avg("retweets"), 2).alias("v")).collect()[0]["v"]
    tweets_per_user_val = tweets.groupBy("user_id").agg(count("tweet_id").alias("c")).agg(spark_round(avg("c"), 2).alias("v")).collect()[0]["v"]

    tweet_metrics = spark.createDataFrame([
        ("total_tweets",          str(total_tweets)),
        ("avg_likes_per_tweet",   str(avg_likes_val)),
        ("avg_retweets_per_tweet",str(avg_retweets_val)),
        ("avg_tweets_per_user",   str(tweets_per_user_val))
    ], ["metric_name", "metric_value"])

    tweets_per_day  = tweets.withColumn("tweet_date", to_date(col("timestamp"))).groupBy("tweet_date").agg(count("tweet_id").alias("tweet_count")).orderBy("tweet_date")
    tweets_per_hour = tweets.withColumn("tweet_hour", hour(col("timestamp"))).groupBy("tweet_hour").agg(count("tweet_id").alias("tweet_count")).orderBy("tweet_hour")

    tweet_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.tweet_metrics")
    tweets_per_day.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.tweets_per_day")
    tweets_per_hour.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.tweets_per_hour")
    logger.info("tweet metric tables written")

    # sentiment metrics
    total_sentiment   = sentiment.count()
    positive_count    = sentiment.filter(col("sentiment_score") > 0.05).count()
    negative_count    = sentiment.filter(col("sentiment_score") < -0.05).count()
    neutral_count     = total_sentiment - positive_count - negative_count
    avg_sentiment_val = sentiment.agg(spark_round(avg("sentiment_score"), 4).alias("v")).collect()[0]["v"]
    pos_pct           = builtins_round(positive_count / total_sentiment * 100, 2) if total_sentiment > 0 else 0
    neg_pct           = builtins_round(negative_count / total_sentiment * 100, 2) if total_sentiment > 0 else 0
    neu_pct           = builtins_round(neutral_count  / total_sentiment * 100, 2) if total_sentiment > 0 else 0

    sentiment_over_time = (
        sentiment
        .withColumn("sentiment_date", to_date(col("tweet_timestamp")))
        .withColumn("sentiment_hour", hour(col("tweet_timestamp")))
        .groupBy("sentiment_date", "sentiment_hour", "topic_category")
        .agg(
            count("tweet_id").alias("total_tweets"),
            spark_round(avg("sentiment_score"), 4).alias("avg_sentiment_score"),
            count(when(col("sentiment_score") > 0.05, 1)).alias("positive_count"),
            count(when(col("sentiment_score") < -0.05, 1)).alias("negative_count"),
            count(when(col("sentiment_score").between(-0.05, 0.05), 1)).alias("neutral_count")
        )
        .orderBy("sentiment_date", "sentiment_hour")
    )

    sentiment_metrics = spark.createDataFrame([
        ("total_tweets",        str(total_sentiment)),
        ("positive_tweets",     str(positive_count)),
        ("negative_tweets",     str(negative_count)),
        ("neutral_tweets",      str(neutral_count)),
        ("positive_pct",        str(pos_pct)),
        ("negative_pct",        str(neg_pct)),
        ("neutral_pct",         str(neu_pct)),
        ("avg_sentiment_score", str(avg_sentiment_val))
    ], ["metric_name", "metric_value"])

    sentiment_metrics.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.sentiment_metrics")
    sentiment_over_time.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.sentiment_over_time")
    logger.info("sentiment metric tables written : sentiment_over_time : " + str(spark.table("social_catalog.gold.sentiment_over_time").count()) + " rows")

    # trend metrics
    top_countries_by_volume = (
        trends.groupBy("country")
        .agg(
            sum("tweet_volume").alias("total_tweet_volume"),
            sum("mention_count").alias("total_mentions"),
            sum("retweet_count").alias("total_retweets"),
            spark_round(avg("trend_score"), 4).alias("avg_trend_score"),
            spark_round(avg("sentiment_index"), 4).alias("avg_sentiment_index")
        )
        .orderBy(desc("total_tweet_volume"))
    )

    trend_over_time = (
        trends
        .withColumn("trend_month", month(col("trend_timestamp")))
        .withColumn("trend_year",  year(col("trend_timestamp")))
        .groupBy("trend_timestamp", "trend_year", "trend_month", "country")
        .agg(
            sum("tweet_volume").alias("total_tweet_volume"),
            sum("mention_count").alias("total_mentions"),
            sum("retweet_count").alias("total_retweets"),
            spark_round(avg("trend_score"), 4).alias("avg_trend_score"),
            spark_round(avg("sentiment_index"), 4).alias("avg_sentiment_index"),
            sum("impressions").alias("total_impressions"),
            sum("engagement_count").alias("total_engagement")
        )
        .orderBy("trend_timestamp")
    )

    top_countries_by_volume.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.trend_metrics")
    trend_over_time.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.trend_over_time")
    logger.info("trend metric tables written : trend_over_time : " + str(spark.table("social_catalog.gold.trend_over_time").count()) + " rows")

    # user metrics
    total_users       = users.count()
    avg_followers_val = users.agg(spark_round(avg("followers_count"), 2).alias("v")).collect()[0]["v"]
    avg_following_val = users.agg(spark_round(avg("following_count"), 2).alias("v")).collect()[0]["v"]

    top_users_by_followers = users.filter(col("followers_count").isNotNull()).select("user_id", "country", "followers_count", "following_count", "posts_count").orderBy(desc("followers_count")).limit(10000)
    user_creation_trend    = users.withColumn("created_date", to_date(col("account_created_date"))).groupBy("created_date", "country", "topic_category").agg(count("user_id").alias("new_users"), spark_round(avg("followers_count"), 2).alias("avg_followers")).orderBy("created_date")
    users_by_country       = users.groupBy("country", "topic_category", "follower_segment").agg(count("user_id").alias("total_users"), spark_round(avg("followers_count"), 2).alias("avg_followers"), spark_round(avg("following_count"), 2).alias("avg_following"), spark_round(avg("posts_count"), 2).alias("avg_posts")).orderBy(desc("total_users")) if "follower_segment" in users.columns else users.groupBy("country", "topic_category").agg(count("user_id").alias("total_users"), spark_round(avg("followers_count"), 2).alias("avg_followers"), spark_round(avg("following_count"), 2).alias("avg_following")).orderBy(desc("total_users"))

    user_summary = spark.createDataFrame([
        ("total_users",   str(total_users)),
        ("avg_followers", str(avg_followers_val)),
        ("avg_following", str(avg_following_val))
    ], ["metric_name", "metric_value"])

    user_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.user_metrics")
    top_users_by_followers.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.top_users_by_followers")
    user_creation_trend.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.user_creation_trend")
    users_by_country.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.users_by_country")
    logger.info("user metric tables written")
    logger.info("user_creation_trend : " + str(spark.table("social_catalog.gold.user_creation_trend").count()) + " rows")
    logger.info("users_by_country    : " + str(spark.table("social_catalog.gold.users_by_country").count()) + " rows")
    logger.info("top_users           : " + str(spark.table("social_catalog.gold.top_users_by_followers").count()) + " rows")

    # valid metrics
    total_valid     = valid.count()
    total_all       = spark.table("social_catalog.gold.fact_tweet").count()
    total_invalid   = total_all - total_valid
    valid_pct_val   = builtins_round(total_valid   / total_all * 100, 2) if total_all > 0 else 0
    invalid_pct_val = builtins_round(total_invalid / total_all * 100, 2) if total_all > 0 else 0

    valid_per_day = (
        valid
        .withColumn("tweet_date", to_date(col("tweet_timestamp")))
        .withColumn("tweet_hour", hour(col("tweet_timestamp")))
        .groupBy("tweet_date", "tweet_hour", "topic_category")
        .agg(
            count("tweet_id").alias("valid_tweet_count"),
            spark_round(avg("sentiment_score"), 4).alias("avg_sentiment_score"),
            sum("impressions").alias("total_impressions"),
            sum("engagement_count").alias("total_engagement")
        )
        .orderBy("tweet_date", "tweet_hour")
    )

    valid_per_topic = (
        valid
        .withColumn("tweet_date", to_date(col("tweet_timestamp")))
        .groupBy("topic_category", "tweet_date")
        .agg(
            count("tweet_id").alias("valid_tweet_count"),
            spark_round(avg("sentiment_score"), 4).alias("avg_sentiment_score"),
            sum("impressions").alias("total_impressions"),
            sum("likes").alias("total_likes")
        )
        .orderBy(desc("valid_tweet_count"))
    )

    valid_summary = spark.createDataFrame([
        ("total_valid_tweets",   str(total_valid)),
        ("total_invalid_tweets", str(total_invalid)),
        ("total_tweets",         str(total_all)),
        ("valid_pct",            str(valid_pct_val)),
        ("invalid_pct",          str(invalid_pct_val))
    ], ["metric_name", "metric_value"])

    valid_summary.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.valid_tweet_metrics")
    valid_per_day.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.valid_tweets_per_day")
    valid_per_topic.write.format("delta").mode("overwrite").option("overwriteSchema", "true").saveAsTable("social_catalog.gold.valid_tweets_per_topic")
    logger.info("valid metric tables written")
    logger.info("valid_tweets_per_day   : " + str(spark.table("social_catalog.gold.valid_tweets_per_day").count()) + " rows")
    logger.info("valid_tweets_per_topic : " + str(spark.table("social_catalog.gold.valid_tweets_per_topic").count()) + " rows")

    # final counts
    logger.info("FINAL GOLD TABLE COUNTS")
    all_gold_tables = [
        "dim_topic", "dim_country", "dim_user",
        "fact_tweet", "fact_trend",
        "agg_sentiment_by_topic", "agg_user_influence",
        "tweet_metrics", "tweets_per_day", "tweets_per_hour",
        "sentiment_metrics", "sentiment_over_time",
        "trend_metrics", "trend_over_time",
        "user_metrics", "top_users_by_followers",
        "user_creation_trend", "users_by_country",
        "valid_tweet_metrics", "valid_tweets_per_day", "valid_tweets_per_topic"
    ]
    for table in all_gold_tables:
        cnt = spark.table("social_catalog.gold." + table).count()
        logger.info("OK : " + table + " | " + str(cnt) + " rows")

except Exception as e:
    logger.error("Gold pipeline failed : " + str(e))
    raise

finally:
    logger.info("Gold pipeline completed")